# Performance Testing - last update; 8/10/2025

This notebook performs performance testing using random individual noise and event files from the test dataset. Most code is borrowed from/inspired by Nikolaj Dahmen (MarsQuakeNet). 

In [ ]:
import os
# Root of the local Bedretto dataset (not distributed with this repo, see README)
DATA_DIR = os.environ.get("BEDRETTO_DATA_DIR", "data") + "/"

In [ ]:
# Comparing performance metrics across different SNR values

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import scipy.signal
import glob
import random
from sklearn.preprocessing import RobustScaler, StandardScaler
import time
from obspy.core import UTCDateTime, Stream, Trace
from obspy.signal.cross_correlation import correlate, xcorr_max

# random seed for reproducibility
#random.seed(42)

plt.rcParams['figure.figsize'] = [12, 10]
plt.rcParams['font.size'] = 12

# test data files 
files_test_event = glob.glob(DATA_DIR + "test_v01/single_waveforms_snr50/test/event/Event[0-9][0-9][0-9][0-9][0-9][0-9].npz")
files_test_noise = glob.glob(DATA_DIR + "test_3dayoffset/single_waveforms_noise_3dayshift/test/noise/Noise[0-9][0-9][0-9][0-9][0-9][0-9].npz")

print(f"Number of test event files: {len(files_test_event)}")
print(f"Number of test noise files: {len(files_test_noise)}")

# model
model_path = DATA_DIR + "models/zcomp_unet_192x64_seventhrun_17_09_2025_trainrep4_valrep4.keras"
try:
    model = tf.keras.models.load_model(model_path)
    print("Model loaded successfully!")
    print(f"Model input shape: {model.input_shape}")
    print(f"Model output shape: {model.output_shape}")
except Exception as e:
    print(f"Error loading model: {e}")

In [ ]:
def normalize_percentile(data, quantile_range=(25,75), unit_variance=False, limit=50, clipping='none', scaler="robust"):
    """
    Normalization of samples 1-component data
    """
    # get data
    data_real = data[:,:,0]  # real
    data_imag = data[:,:,1]  # imag

    freq_bins, time_bins = data_real.shape  
    # define scaler
    if scaler == "robust":
        scaler_obj_real = RobustScaler(unit_variance=unit_variance)
        scaler_obj_imag = RobustScaler(unit_variance=unit_variance)
    else:
        scaler_obj_real = StandardScaler()
        scaler_obj_imag = StandardScaler()

    # apply to data: first flatten and then reshape
    data_real_norm = scaler_obj_real.fit_transform(np.expand_dims(data_real.flatten(),1)).reshape((freq_bins, time_bins))
    data_imag_norm = scaler_obj_imag.fit_transform(np.expand_dims(data_imag.flatten(),1)).reshape((freq_bins, time_bins))

    if clipping=='constant':  # hard clipping, all values > limit
        data_real_norm[data_real_norm>limit]=limit
        data_real_norm[data_real_norm<-limit]=-limit
        data_imag_norm[data_imag_norm>limit]=limit
        data_imag_norm[data_imag_norm<-limit]=-limit

    # combine real and imag
    data_return = np.zeros(data.shape)
    data_return[:,:,0] = data_real_norm
    data_return[:,:,1] = data_imag_norm

    return data_return, {"scaler_real": scaler_obj_real, "scaler_imag": scaler_obj_imag}

def map_func_v2(event_file, noise_file, force_event, target_snr=None, clip_val=1000, channel_norm=True, sps=100000, scaler="robust"):

    len_sample = 4967
    noise_tmp = np.load(noise_file, allow_pickle=True) 
    min_snr = 0.2 
    delta = 0
    ampl_fac = None 
    
    stft_parameters = {"nperseg": 64, "nfft": 126, "fs": 100000, "noverlap": 38, "boundary": "zeros", "padded": False}

    # AMOUNT OF EVENTS (0, 1, or 2) AND TIME SHIFT 
    if force_event is True:
            # Force event
            ev_tmp = np.load(event_file, allow_pickle=True)
            ev_selected, second_event = True, False
    elif force_event is False:
        # Force no event
        ev_tmp = {}
        ev_tmp["processed_waveform_Z"] = np.zeros(8000)
        ev_selected, second_event = False, False
    else:
        # Original random logic
        if random.random() < 0.5: # 1 EVENT #0.5
            ev_tmp = np.load(event_file, allow_pickle=True)
            ev_selected, second_event = True, False
        else:
            if random.random() < 1:  # NO EVENT #0.9
                ev_tmp = {}
                ev_tmp["processed_waveform_Z"] = np.zeros(8000) # event is 8000 samples long
                ev_selected, second_event = False, False
            else:  # 2 EVENTS
                ev_tmp = np.load(event_file, allow_pickle=True) # event is 8000 samples long
                ev_selected, second_event = True, True
                delta = int(sps * random.uniform(0.01, 0.02500))
                ampl_fac = random.uniform(0.1, 0.9)
    
    # SNR SELECTION
    if target_snr is not None:
        final_snr = target_snr
    else:
        if random.random() < 0.33:
            final_snr = random.uniform(min_snr / 2, 100) 
        else:
            if random.random() < 0.5:
                final_snr = random.uniform(min_snr / 2, min_snr)
            else:
                final_snr = random.uniform(min_snr, 1)
    
    snr_ev_1C, std_noise_1C = [], []
    ev_data, noise_data = {}, {}

    comp = "Z"
    noise_snr = noise_tmp["snr"]
    noise_comp = noise_tmp["noise_waveform_" + comp]

    # shift noise start between 0 and 2000 samples
    noise_shift = int(random.uniform(0, 1499))
    noise_comp = noise_comp[noise_shift:noise_shift + len_sample]

    # process event component
    ev_comp = random.choice([-1, 1]) * ev_tmp["processed_waveform_Z"]

    # random time shift
    shift_event = int(random.uniform(0, 2999))

    # Second event adding process
    if second_event:
        # Adjust delta polarity
        delta = int(np.sign(shift_event - 0.0149 * sps) * delta)

        # Create padded array (original + padding for second event)
        padded_length = len(ev_comp) + 5000
        ev_comp_padded = np.zeros(padded_length)
        
        # Place first event
        ev_comp_padded[2500:2500 + len(ev_comp)] = ev_comp

        # Place second event at delta offset
        ev_comp_padded[2500+delta:2500+delta + len(ev_comp)] += ampl_fac * ev_comp
        # Extract the relevant segment
        ev_comp = ev_comp_padded[2500 + shift_event: 2500 + shift_event + len_sample]
    else:
        ev_comp = ev_comp[shift_event:shift_event + len_sample]
    
    # compute STFTs
    f, t, ev_stft_comp = scipy.signal.stft(ev_comp, **stft_parameters)
    _, _, noise_stft_comp = scipy.signal.stft(noise_comp, **stft_parameters)
    
    ev_data["stft_" + comp] = ev_stft_comp
    noise_data["stft_" + comp] = noise_stft_comp

    # calculate SNR 
    if ev_selected and np.any(np.abs(ev_stft_comp) > 1e-10):
        snr_ev_1C.append(np.std(ev_stft_comp))
        std_noise_1C.append(np.std(noise_stft_comp))
        original_snr = np.mean(snr_ev_1C) / (np.mean(std_noise_1C) + 1e-10)
    else:
        original_snr = 1e-10

    # scale event to achieve target SNR
    if ev_selected and original_snr > 1e-8:
        ev_stft_comp_scaled = (ev_data["stft_" + comp] / original_snr) * final_snr
    else:
        ev_stft_comp_scaled = ev_data["stft_" + comp]

    noise_stft_comp_scaled = noise_data["stft_" + comp]

    # combine event and noise
    noisy_signal = ev_stft_comp_scaled + noise_stft_comp_scaled

    # compute mask
    ev_magnitude = np.abs(ev_stft_comp_scaled)
    noise_magnitude = np.abs(noise_stft_comp_scaled)
    
    # add small epsilon to avoid division by zero
    epsilon = 1e-40
    tmp_mask = ev_magnitude / (ev_magnitude + noise_magnitude + epsilon)
    
    tmp_mask[tmp_mask >= 1] = 1  # safety check
    tmp_mask[tmp_mask <= 0] = 0  # safety check

    # check for NaN/inf
    if (np.sum(np.isnan(tmp_mask)) > 0 or np.sum(np.isnan(noisy_signal)) > 0 or
            np.sum(np.isinf(tmp_mask)) > 0 or np.sum(np.isinf(noisy_signal)) > 0):
        print('Warning: NaN/inf detected, replacing with zeros')
        noisy_signal = np.zeros_like(noisy_signal)
        tmp_mask = np.zeros_like(tmp_mask)

    # save data
    dict_save = {}
    dict_save["noisy_signal_" + comp] = np.stack([noisy_signal.real, noisy_signal.imag], axis=-1).astype('float32')
    dict_save["mask" + comp] = tmp_mask.astype('float32')

    if channel_norm:
        noisy_signal_Z_raw, norm_scalers = normalize_percentile(dict_save['noisy_signal_Z'], scaler=scaler)
        noisy_signal_Z = noisy_signal_Z_raw.astype('float32')
    else:
        noisy_signal_Z = dict_save['noisy_signal_Z']

    mask_Z_temp = dict_save['maskZ'].astype('float32')
    mask_Z = np.zeros([mask_Z_temp.shape[0], mask_Z_temp.shape[1], 1], dtype='float32')
    mask_Z[:, :, 0] = mask_Z_temp

    mask = mask_Z

    norm_params = {
        'scalers': norm_scalers,
        'original_snr': original_snr,
        'final_snr': final_snr
    }

    # Get actual P-arrival time in seconds
    p_arrival_time = (3000 - shift_event) / sps
    
    return noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, noisy_signal, ev_stft_comp, noise_stft_comp, norm_params, ev_selected, shift_event, p_arrival_time

In [ ]:
def select_random_files():
    if len(files_test_event) == 0 or len(files_test_noise) == 0:
        print("No test files available")
        return None, None
    
    event_file = random.choice(files_test_event)
    noise_file = random.choice(files_test_noise)
    
    return event_file, noise_file


def calculate_metrics(predicted_mask_2d, ev_selected, threshold=0.5):
    """
    Calc precision, recall and F1 score
   
    """
    # get global max
    global_max = np.max(predicted_mask_2d)
    prediction = 1 if global_max >= threshold else 0
    actual = 1 if ev_selected else 0
    
    # Calculate TP, FP, FN, TN for this single sample
    TP = 1 if (prediction == 1) and (actual == 1) else 0
    FP = 1 if (prediction == 1) and (actual == 0) else 0
    FN = 1 if (prediction == 0) and (actual == 1) else 0
    TN = 1 if (prediction == 0) and (actual == 0) else 0
    
    return TP, FP, FN, TN

def evaluate_model_performance_denoise(model, target_snr, num_samples=20, threshold=0.7):
    """
    Evaluate denoising model performance at a specific SNR 
    """
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0

    # # Ensure exactly 50/50 split
    # num_events = num_samples // 2
    # num_noise = num_samples - num_events
    # 30% events, 70% noise
    num_events = int(num_samples * 0.8)  # 80/20 split
    num_noise = num_samples - num_events  
    
    for i in range(num_events):
        event_file, noise_file = select_random_files()
        if event_file is None or noise_file is None:
            continue
        
        # data generation with forced event
        (noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, 
         noisy_signal, ev_stft_comp, noise_stft_comp, norm_params, 
         ev_selected, shift_event, p_arrival_time) = map_func_v2(
            event_file, noise_file, target_snr=target_snr, force_event=True)
        
        # model prediction
        model_input = np.expand_dims(noisy_signal_Z, axis=0)
        with tf.device('/gpu:0'):
            predicted_mask = model.predict(model_input, verbose=0)
        
        predicted_mask_2d = predicted_mask[0, :, :, 0]
        
        # Calc metrics 
        TP, FP, FN, TN = calculate_metrics(predicted_mask_2d, ev_selected, threshold)
        
        true_positives += TP
        false_positives += FP
        false_negatives += FN
        true_negatives += TN

    for i in range(num_noise):
        event_file, noise_file = select_random_files()
        if event_file is None or noise_file is None:
            continue
        
        # force no event
        (noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, 
         noisy_signal, ev_stft_comp, noise_stft_comp, norm_params, 
         ev_selected, shift_event, p_arrival_time) = map_func_v2(
            event_file, noise_file, target_snr=target_snr, force_event=False)
        
        # get model prediction
        model_input = np.expand_dims(noisy_signal_Z, axis=0)
        with tf.device('/gpu:0'):
            predicted_mask = model.predict(model_input, verbose=0)
        
        predicted_mask_2d = predicted_mask[0, :, :, 0]
        TP, FP, FN, TN = calculate_metrics(predicted_mask_2d, ev_selected, threshold)
        
        true_positives += TP
        false_positives += FP
        false_negatives += FN
        true_negatives += TN
    
    # calc precision, recall, and F1 score from accumulated counts
    precision = true_positives / (true_positives + false_positives + 1e-10)
    recall = true_positives / (true_positives + false_negatives + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    
    print(f"SNR {target_snr:.3f} - Denoising: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")
    print(f"TP={true_positives}, FP={false_positives}, FN={false_negatives}, TN={true_negatives}")
    
    return precision, recall, f1, true_positives, false_positives, false_negatives, true_negatives



In [ ]:
from scipy.interpolate import make_interp_spline

def run_performance_comparison(model, snr_values, num_samples=10, threshold=0.7):
    """
    Run performance comparison between denoising and bandpass filtering across multiple SNR values
    """
    # Initialize result dictionaries
    results_denoise = {
        'precision': [],
        'recall': [],
        'f1': [],
        'tp': [],
        'fp': [],
        'fn': [],
        'tn': []
    }

    # each SNR value
    for snr in snr_values:
        print(f"\n=== Evaluating SNR: {snr:.3f} ===")
        
        p_denoise, r_denoise, f1_denoise, tp_denoise, fp_denoise, fn_denoise, tn_denoise = evaluate_model_performance_denoise(
            model, snr, num_samples=num_samples, threshold=threshold)

        # Store results 
        results_denoise['precision'].append(p_denoise)
        results_denoise['recall'].append(r_denoise)
        results_denoise['f1'].append(f1_denoise)
        results_denoise['tp'].append(tp_denoise)
        results_denoise['fp'].append(fp_denoise)
        results_denoise['fn'].append(fn_denoise)
        results_denoise['tn'].append(tn_denoise)
    
    return {
        'snr_values': snr_values,
        'denoise': results_denoise,
    }

def plot_performance_metrics(results):
    """
    Plot performance metrics for denoising
    """
    snr_values = results['snr_values']  # assumed in linear scale (0.03 to 3.16)
    denoise_results = results['denoise']
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    #smooth curves
    snr_dense = np.logspace(np.log10(snr_values[0]), np.log10(snr_values[-1]), 200)

    # denoising
    precision_spline = make_interp_spline(snr_values, denoise_results['precision'], k=1)
    recall_spline = make_interp_spline(snr_values, denoise_results['recall'], k=1)
    f1_spline = make_interp_spline(snr_values, denoise_results['f1'], k=1)

    ax.plot(snr_dense, precision_spline(snr_dense), 'b', label='Denoising - Precision', linewidth=2)
    ax.plot(snr_dense, recall_spline(snr_dense), 'g', label='Denoising - Recall', linewidth=2)
    ax.plot(snr_dense, f1_spline(snr_dense), 'r', label='Denoising - F1 Score', linewidth=2)

    # original points for reference
    ax.plot(snr_values, denoise_results['precision'], 'bo', markersize=8)
    ax.plot(snr_values, denoise_results['recall'], 'go', markersize=8)
    ax.plot(snr_values, denoise_results['f1'], 'ro', markersize=8)
    
    # labels
    ax.set_xlabel(r'SNR $\sigma_{event}/\sigma_{noise}$', fontsize=14)
    ax.set_ylabel('Score', fontsize=14)
    ax.legend(fontsize=12)
    ax.set_ylim(0, 1.05)
    
    ax.set_xscale("log")

    handles, labels = ax.get_legend_handles_labels()
    handles = [plt.Line2D([0], [0], color='white', linewidth=0)] + handles
    labels = ['Performance Metrics \n(#/SNR 500, threshold 0.247)'] + labels
    ax.legend(handles, labels, fontsize=10, loc='lower right')
    xticks = [0.03, 0.1, 0.32, 1.0, 3.16]
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(x) for x in xticks])
    ax.set_xlim(0.03, 3.3)

    # # bottom axis: SNR [dB]
    # ax2 = ax.twiny()
    # ax2.set_xlim(-15, 5)
    # # Map SNR values to dB for the top axis
    # db_ticks = 10 * np.log10(np.array( xticks))
    # ax2.set_xticks(xticks)
    # ax2.set_xticklabels([f"{db:.0f}" for db in db_ticks])
    # ax2.set_xlabel('SNR [dB]', fontsize=14)
    # ax2.xaxis.set_label_position('bottom')
    # ax2.xaxis.set_ticks_position('bottom')
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ax.grid(True, which="major", axis="both", alpha=0.3)
    
    fig.tight_layout()
    plt.show()

In [ ]:
snr_values = np.logspace(np.log10(0.03), np.log10(3.16), num=11)
print(snr_values)

In [ ]:
snr_values = np.logspace(np.log10(0.03), np.log10(3.16), num=11)
print(snr_values)
#  num_samples
num_samples_per_snr = 500  

start_time = time.time()

results = run_performance_comparison(model, snr_values, num_samples=num_samples_per_snr, threshold=0.247)

end_time = time.time()
print(f"\ncompleted in {end_time - start_time:.2f} seconds")

plot_performance_metrics(results)

denoise_results = results['denoise']

print("\nDenoising Performance:")
print(f"  Avg Precision: {np.mean(denoise_results['precision']):.3f}")
print(f"  Avg Recall: {np.mean(denoise_results['recall']):.3f}")
print(f"  Avg F1-Score: {np.mean(denoise_results['f1']):.3f}")


In [ ]:
plot_performance_metrics(results)

In [ ]:
def find_optimal_threshold_roc(model, snr_values, num_samples=100):
    """
    Find optimal threshold using ROC curve analysis across all SNR values
    """
    all_predictions = []
    all_ground_truth = []
    
    for snr in snr_values:
        print(f"Collecting data for SNR {snr:.3f}...")
        
        # Use balanced dataset (50/50 events/noise)
        num_events = num_samples // 2
        num_noise = num_samples - num_events
        
        # Process events
        for i in range(num_events):
            event_file, noise_file = select_random_files()
            if event_file is None or noise_file is None:
                continue
            
            (noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, 
             noisy_signal, ev_stft_comp, noise_stft_comp, norm_params, 
             ev_selected, shift_event, p_arrival_time) = map_func_v2(
                event_file, noise_file, target_snr=snr, force_event=True)
            
            model_input = np.expand_dims(noisy_signal_Z, axis=0)
            with tf.device('/gpu:0'):
                predicted_mask = model.predict(model_input, verbose=0)
            predicted_mask_2d = predicted_mask[0, :, :, 0]
            global_max = np.max(predicted_mask_2d)
            
            all_predictions.append(global_max)
            all_ground_truth.append(1)  # Event present
        
        # Process noise
        for i in range(num_noise):
            event_file, noise_file = select_random_files()
            if event_file is None or noise_file is None:
                continue
            
            (noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, 
             noisy_signal, ev_stft_comp, noise_stft_comp, norm_params, 
             ev_selected, shift_event, p_arrival_time) = map_func_v2(
                event_file, noise_file, target_snr=snr, force_event=False)
            
            model_input = np.expand_dims(noisy_signal_Z, axis=0)
            with tf.device('/gpu:0'):
                predicted_mask = model.predict(model_input, verbose=0)
            
            predicted_mask_2d = predicted_mask[0, :, :, 0]
            global_max = np.max(predicted_mask_2d)
            
            all_predictions.append(global_max)
            all_ground_truth.append(0)  # No event
    
    # Convert to numpy arrays
    all_predictions = np.array(all_predictions)
    all_ground_truth = np.array(all_ground_truth)
    
    print(f"Total samples collected: {len(all_predictions)}")
    print(f"Events: {np.sum(all_ground_truth)}, Noise: {len(all_ground_truth) - np.sum(all_ground_truth)}")
    
    # Test different threshold values
    thresholds = np.linspace(0.01, 0.95, 100)
    tpr_values = []  # True Positive Rate (Sensitivity/Recall)
    fpr_values = []  # False Positive Rate (1 - Specificity)
    f1_values = []   # F1 scores
    
    for threshold in thresholds:
        predictions = (all_predictions >= threshold).astype(int)
        
        tp = np.sum((predictions == 1) & (all_ground_truth == 1))
        fp = np.sum((predictions == 1) & (all_ground_truth == 0))
        fn = np.sum((predictions == 0) & (all_ground_truth == 1))
        tn = np.sum((predictions == 0) & (all_ground_truth == 0))
        
        tpr = tp / (tp + fn + 1e-10)  # Recall
        fpr = fp / (fp + tn + 1e-10)  # False Positive Rate
        precision = tp / (tp + fp + 1e-10)
        f1 = 2 * precision * tpr / (precision + tpr + 1e-10)
        
        tpr_values.append(tpr)
        fpr_values.append(fpr)
        f1_values.append(f1)
    
    tpr_values = np.array(tpr_values)
    fpr_values = np.array(fpr_values)
    f1_values = np.array(f1_values)
    
    # max f1 score
    optimal_idx_f1 = np.argmax(f1_values)
    optimal_threshold_f1 = thresholds[optimal_idx_f1]
    
    # Use F1-based threshold as default
    optimal_threshold = optimal_threshold_f1
    
    return optimal_threshold, {
        'thresholds': thresholds,
        'tpr': tpr_values,
        'fpr': fpr_values,
        'f1': f1_values,
        'optimal_f1': optimal_threshold_f1,
        'all_predictions': all_predictions,
        'all_ground_truth': all_ground_truth
    }

def plot_roc_analysis(roc_data):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))
    
    # F1 Score vs Threshold
    ax1.plot(roc_data['thresholds'], roc_data['f1'], 'b-', linewidth=2)
    ax1.axvline(roc_data['optimal_f1'], color='r', linestyle='--', 
                label=f'Optimal F1: {roc_data["optimal_f1"]:.3f}')
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('F1 Score')
    ax1.set_title('F1 Score vs Threshold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Prediction distribution
    events_preds = roc_data['all_predictions'][roc_data['all_ground_truth'] == 1]
    noise_preds = roc_data['all_predictions'][roc_data['all_ground_truth'] == 0]
    
    ax2.hist(noise_preds, bins=50, alpha=0.7, label='Noise', color='red', density=True)
    ax2.hist(events_preds, bins=50, alpha=0.7, label='Events', color='blue', density=True)
    ax2.axvline(roc_data['optimal_f1'], color='black', linestyle='--', 
                label=f'Optimal Threshold: {roc_data["optimal_f1"]:.3f}')
    ax2.set_xlabel('Prediction Score')
    ax2.set_ylabel('Density')
    ax2.set_title('Prediction Score Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Define SNR range for optimization
snr_values_opt = np.logspace(np.log10(0.03), np.log10(3.16), num=7)
print(f"SNR values for optimization: {snr_values_opt}")

# Method 1: ROC-based optimization (recommended)
print("=== ROC-BASED OPTIMIZATION ===")
optimal_threshold_roc, roc_data = find_optimal_threshold_roc(
    model, snr_values_opt, num_samples=50)

# Plot ROC analysis
plot_roc_analysis(roc_data)


#cross correlation 


In [ ]:
def calculate_cross_correlation(signal1, signal2):
    """
    Calculate zero-lag cross-correlation between two signals
    """
    # normalize signals
    signal1_norm = (signal1 - np.mean(signal1)) / (np.std(signal1) + 1e-10)
    signal2_norm = (signal2 - np.mean(signal2)) / (np.std(signal2) + 1e-10)
    
    # calculate zero-lag cross-correlation
    cross_corr = np.mean(signal1_norm * signal2_norm)
    
    return cross_corr

def bandpass_filter_signal(signal, fs=100000, lowcut=200, highcut=7500, order=5):
    """
    Apply bandpass filter to signal (both causal and acausal)
    """
    from scipy.signal import butter, filtfilt, lfilter
    
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    
    # butterworth bandpass filter
    b, a = butter(order, [low, high], btype='band')
    
    #forward
    filtered_causal = lfilter(b, a, signal)

    # backward
    filtered_acausal = filtfilt(b, a, signal)
    
    return filtered_causal, filtered_acausal

In [ ]:
def evaluate_cross_correlation_at_snr(model, target_snr, num_samples=500):
    """
    Evaluate cross-correlations at a specific SNR
    """
    denoised_corrs = []
    bandpass_causal_corrs = []
    bandpass_acausal_corrs = []
    noisy_corrs = []

    print(f"\n SNR {target_snr:.3f} with {num_samples} samples...")
    for i in range(num_samples):
        if i % 100 == 0:
            print(f"sample {i+1}/{num_samples}")
        
        event_file, noise_file = select_random_files()
        if event_file is None or noise_file is None:
            continue
        
        #mapping
        (noisy_signal_Z, mask, ev_comp, noise_comp, final_snr, 
         noisy_signal_stft, ev_stft_comp, noise_stft_comp, norm_params, 
         ev_selected, shift_event, p_arrival_time) = map_func_v2(
            event_file, noise_file, target_snr=target_snr, force_event=True)
        
        if not ev_selected:
            continue  # skip if no event
        
        original_event = ev_comp  # This is the clean event signal
        
        # Get noisy signal in time domain (convert from STFT)
        _, noisy_time_signal = scipy.signal.istft(
            noisy_signal_stft, 
            nperseg=64, noverlap=38, nfft=126, fs=100000, 
            boundary=False, input_onesided=False)
        
        # Ensure same length as original event
        min_len = min(len(original_event), len(noisy_time_signal))
        original_event = original_event[:min_len]
        noisy_time_signal = noisy_time_signal[:min_len]
        
        # denoised signal
        model_input = np.expand_dims(noisy_signal_Z, axis=0)
        with tf.device('/gpu:0'):
            predicted_mask = model.predict(model_input, verbose=0)
        
        # apply predicted mask 
        predicted_mask_2d = predicted_mask[0, :, :, 0]
        denoised_stft = noisy_signal_stft * predicted_mask_2d
        
        # convert denoised STFT back to time domain
        _, denoised_time_signal = scipy.signal.istft(
            denoised_stft,
            nperseg=64, noverlap=38, nfft=126, fs=100000,
            boundary=False, input_onesided=False)
        
        denoised_time_signal = denoised_time_signal[:min_len]
        
        #bandpass filters to noisy signal
        bandpass_causal, bandpass_acausal = bandpass_filter_signal(
            noisy_time_signal, fs=100000, lowcut=200, highcut=7500)
        
        bandpass_causal = bandpass_causal[:min_len]
        bandpass_acausal = bandpass_acausal[:min_len]
        
        # calculate cross-correlations
        try:
            # Denoised vs original
            corr_denoised = calculate_cross_correlation(denoised_time_signal, original_event)
            denoised_corrs.append(corr_denoised)
            
            # Bandpass causal vs original
            corr_bp_causal = calculate_cross_correlation(bandpass_causal, original_event)
            bandpass_causal_corrs.append(corr_bp_causal)
            
            # Bandpass acausal vs original
            corr_bp_acausal = calculate_cross_correlation(bandpass_acausal, original_event)
            bandpass_acausal_corrs.append(corr_bp_acausal)
            
            # Noisy vs original
            corr_noisy = calculate_cross_correlation(noisy_time_signal, original_event)
            noisy_corrs.append(corr_noisy)
            
        except Exception as e:
            print(f"Error calculating correlation for sample {i}: {e}")
            continue
    
    # median values
    results = {
        'median_denoised': np.median(denoised_corrs) if denoised_corrs else 0,
        'median_bandpass_causal': np.median(bandpass_causal_corrs) if bandpass_causal_corrs else 0,
        'median_bandpass_acausal': np.median(bandpass_acausal_corrs) if bandpass_acausal_corrs else 0,
        'median_noisy': np.median(noisy_corrs) if noisy_corrs else 0,
        'num_valid_samples': len(denoised_corrs)
    }
    
    print(f"  Valid samples: {results['num_valid_samples']}")
    print(f"  Median correlations - Denoised: {results['median_denoised']:.3f}, "
          f"BP Causal: {results['median_bandpass_causal']:.3f}, "
          f"BP Acausal: {results['median_bandpass_acausal']:.3f}, "
          f"Noisy: {results['median_noisy']:.3f}")
    
    return results

In [ ]:
def run_cross_correlation_analysis(model, snr_values, num_samples=500):
    """
    run cross-correlation analysis across multiple SNR values
    """
    results = {
        'snr_values': snr_values,
        'denoised': [],
        'bandpass_causal': [],
        'bandpass_acausal': [],
        'noisy': [],
        'valid_samples': []
    }
 
    print(f"SNR range: {snr_values[0]:.3f} to {snr_values[-1]:.3f}")
    print(f"Samples per SNR: {num_samples}")
    
    for snr in snr_values:
        print(f"SNR:{snr:.3f}")
        
        snr_results = evaluate_cross_correlation_at_snr(
            model, snr, num_samples=num_samples)
        
        results['denoised'].append(snr_results['median_denoised'])
        results['bandpass_causal'].append(snr_results['median_bandpass_causal'])
        results['bandpass_acausal'].append(snr_results['median_bandpass_acausal'])
        results['noisy'].append(snr_results['median_noisy'])
        results['valid_samples'].append(snr_results['num_valid_samples'])
    
    return results

In [ ]:
def plot_cross_correlation_results(results):
    """
    cross-correlation results vs SNR
    """
    snr_values = results['snr_values']
    
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # smooth curves
    snr_dense = np.logspace(np.log10(snr_values[0]), np.log10(snr_values[-1]), 200)
    
    # interpolate
    denoised_spline = make_interp_spline(snr_values, results['denoised'], k=1)
    bp_causal_spline = make_interp_spline(snr_values, results['bandpass_causal'], k=1)
    bp_acausal_spline = make_interp_spline(snr_values, results['bandpass_acausal'], k=1)
    noisy_spline = make_interp_spline(snr_values, results['noisy'], k=1)
    
    # smooth curves
    ax.plot(snr_dense, denoised_spline(snr_dense), 'b-', 
            label='Denoised Signal', linewidth=2.5)
    ax.plot(snr_dense, bp_acausal_spline(snr_dense), 'g-', 
            label='Bandpass Acausal (200-7500 Hz)', linewidth=2.5)
    ax.plot(snr_dense, bp_causal_spline(snr_dense), 'orange', linestyle='--',
            label='Bandpass Causal (200-7500 Hz)', linewidth=2)
    ax.plot(snr_dense, noisy_spline(snr_dense), 'r-', 
            label='Noisy Signal', linewidth=2)
    
    # original data points
    ax.plot(snr_values, results['denoised'], 'bo', markersize=8, alpha=0.8)
    ax.plot(snr_values, results['bandpass_acausal'], 'go', markersize=8, alpha=0.8)
    ax.plot(snr_values, results['bandpass_causal'], 'o', color='orange', markersize=8, alpha=0.8)
    ax.plot(snr_values, results['noisy'], 'ro', markersize=8, alpha=0.8)
    
    # formatting
    ax.set_xlabel(r'SNR $\sigma_{event}/\sigma_{noise}$', fontsize=14)
    ax.set_ylabel('Cross-Correlation (Zero-Lag)', fontsize=14)
    ax.set_title('Cross-Correlation with Original Event Signal', fontsize=16)
    
    xticks = [0.03, 0.1, 0.32, 1.0, 3.16]
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(x) for x in xticks])
    ax.set_xlim(0.03, 3.3)
    
    # Y-axis limits
    ax.set_ylim(-0.1, 1.05)
    
    # Legend
    ax.legend(fontsize=12, loc='lower right')
    
    # Grid
    ax.grid(True, which="major", axis="both", alpha=0.3)
    
    # Move x-axis to top
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n=== CROSS-CORRELATION SUMMARY ===")
    print(f"Average Cross-Correlations:")
    print(f"  Denoised: {np.mean(results['denoised']):.3f}")
    print(f"  Bandpass Acausal: {np.mean(results['bandpass_acausal']):.3f}")
    print(f"  Bandpass Causal: {np.mean(results['bandpass_causal']):.3f}")
    print(f"  Noisy: {np.mean(results['noisy']):.3f}")

In [ ]:
# Define SNR values (same as performance metrics)
snr_values = np.logspace(np.log10(0.03), np.log10(3.16), num=11)
print(f"SNR values for cross-correlation analysis: {snr_values}")

# Run cross-correlation analysis
print("Starting cross-correlation analysis...")
start_time = time.time()

cross_corr_results = run_cross_correlation_analysis(
    model, snr_values, num_samples=10)

end_time = time.time()
print(f"\nCross-correlation analysis completed in {end_time - start_time:.2f} seconds")

# Plot results
plot_cross_correlation_results(cross_corr_results)